# AstroGuide — Phase 2 Demo (Google Colab)

This notebook demonstrates the two LangChain tools wired to a ReAct agent with memory,
plus Pydantic-parsed structured output.

**Prerequisites**: Add your `GOOGLE_API_KEY` to Colab Secrets (🔑 icon in the left sidebar).

In [ ]:
# Cell 1 — Clone the repo & install dependencies
!git clone https://github.com/adityarana2610/AstroGuide.git
%cd AstroGuide
!pip install -r requirements.txt -q

In [ ]:
# Cell 2 — Set API key from Colab Secrets & add project to path
import os, sys
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Ensure src/ is importable
if "/content/AstroGuide" not in sys.path:
    sys.path.insert(0, "/content/AstroGuide")

from src.agent import ask, parse_chart_summary
from src.schemas import ChartSummary

print("Imports OK ✅")

In [ ]:
# Cell 3 — Demo question (triggers BOTH tools: birth chart + numerology)
result = ask(
    "My name is Aditya, born 1998-05-14 at 09:30 in Mumbai, India. "
    "What's my Moon sign and my life path number?"
)

# Print the final AI response
print(result["messages"][-1].content)

In [ ]:
# Cell 4 — Raw message trace (SCREENSHOT THIS — proof of tool_calls)
for i, msg in enumerate(result["messages"]):
    print(f"\n{'='*60}")
    print(f"Message {i}: {msg.__class__.__name__}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"  tool_calls: {msg.tool_calls}")
    if hasattr(msg, 'content'):
        print(f"  content: {msg.content[:500]}")

In [ ]:
# Cell 5 — Pydantic structured output demo
from src.tools import CHART_CACHE

# Build a prompt from cached chart data
chart_data = CHART_CACHE.get("Aditya", {})
prompt = (
    f"Extract a ChartSummary from this Vedic birth chart data.\n"
    f"Name: Aditya\n"
    f"Chart positions: {chart_data}\n"
    f"Return the name, moon_sign, ascendant, and sun_sign."
)

summary: ChartSummary = parse_chart_summary(prompt)
print(f"\nPydantic-parsed output:")
print(f"  Type : {type(summary).__name__}")
print(f"  Name : {summary.name}")
print(f"  Moon : {summary.moon_sign}")
print(f"  Asc  : {summary.ascendant}")
print(f"  Sun  : {summary.sun_sign}")